# Core machine learning · First steps

**Level** 1 · **Time** about 20-30 minutes

>We will explore some core concepts of machine learning, starting with the fundamental question: What even is machine learning and why do we do it?
By the end of this lesson, you will hopefully see the vision of machine learning as a tool to help us make better decisions and predictions based on data.

To learn from this notebook, you just have to run it. You do not need to change or understand any of the code.

## Help a busy student

Marc is an archeology student. Working with his professor, they discovered hundreds of pottery fragments in a dig site. Preliminary analysis showed that the fragments belong to two ancient Roman vases, which we will call Vase A and Vase B. The professor wants to know which fragments belong to which vase, so she can reconstruct the vases, but she is too busy to do the work herself. Each fragment has to be inspected under a microscope to determine which vase it belongs to. So the professor passes this tedious and time-consuming task to Marc. Today is Friday, and the professor wants the fragments sorted by Monday, so she can start reconstructing the vases. 

But Marc is busy. He already has plans for the weekend, as he is going to a music festival with his friends. In order to quickly classify the fragments, he sorts only 100 of them and then plans on using a machine learning model to classify the rest 400. He assumes that the position of the fragments in the dig site must be a good indicator of their type, as fragments from the same vase are likely to be found in similar locations, so he also records the coordinates of each fragment. 

The problem is that Marc is an archeology student, so he knows nothing about machine learning, so he asks you to help him. Help him as best as you can, knowing that the more you will be wrong, the more will Marc be in trouble with his professor.

In [ ]:
#@title Run this cell (click ▶). It loads the data and the tools. You do not need to read the code.
# Section for the imports. Imports are libraries that contain useful functions and classes that we can use in our code.
import io
import json
import uuid

import ipywidgets as widgets
import matplotlib.pyplot as plt     # Matplotlib library for plotting
import numpy as np                  # Numpy library for numerical computing
from IPython.display import HTML, display
from sklearn.datasets import make_moons     # Scikit-learn library is the major machine learning library in Python. You will use it a lot.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 42
CLASS_COLORS = ["#386cb0", "#f0027f"]
classes = ["Vase A", "Vase B"]      # Fragment classes. Class 0 is Vase A and class 1 is Vase B.

# make_moons generates the data for you. X is a 2D array of shape (100, 2) containing the coordinates of the fragments,
# and y is a 1D array of shape (100,) containing the labels (0 or 1) for each fragment.
X, y = make_moons(n_samples=100, noise=0.3, random_state=SEED)

# The 400 fragments Marc did not sort himself, with their true labels. We only look at them at the very end.
X_new, y_new = make_moons(n_samples=400, noise=0.3, random_state=SEED + 1)

PAD = 0.35  # how far the plots reach beyond the outermost fragment
XLIM = (X[:, 0].min() - PAD, X[:, 0].max() + PAD)
YLIM = (X[:, 1].min() - PAD, X[:, 1].max() + PAD)


# --------------------------------------------------
# Plots drawn with matplotlib
# --------------------------------------------------

def plot_dataset(X, y, ax, marker="o", alpha=0.8, classes=classes):
    """Draw the fragments as dots, coloured by the vase they belong to."""
    for label in (0, 1):
        ax.scatter(X[y == label, 0], X[y == label, 1], c=CLASS_COLORS[label],
                   label=classes[label], alpha=alpha, marker=marker)
    ax.set(xlabel="x", ylabel="y", xlim=XLIM, ylim=YLIM)
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.4)
    return ax


def plot_decision_boundary(model, ax):
    """Colour every place in the dig site by the vase the model predicts there."""
    # First we construct a grid of points ...
    xx, yy = np.meshgrid(np.linspace(*XLIM, 250), np.linspace(*YLIM, 250))
    # ... then we ask the model which vase each point belongs to ...
    prediction = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    # ... and colour the points by the predicted vase.
    ax.contourf(xx, yy, prediction, levels=[-.5, .5, 1.5], colors=CLASS_COLORS, alpha=.18)


def show_dataset():
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_dataset(X, y, ax)
    ax.set_title("Position of pottery fragments in the dig site and the vase they belong to")
    plt.show()


def _png(fig):
    """The figure as PNG bytes, for an image widget. Drawing into an image widget works the same in Colab, VS Code and Jupyter."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


def _picture_widget():
    return widgets.Image(format="png", layout=widgets.Layout(width="680px", max_width="100%", height="auto"))


# --------------------------------------------------
# Models
# --------------------------------------------------

def polynomial_logistic_regression(degree):
    """Logistic regression whose boundary is a polynomial of the given degree."""
    return make_pipeline(
        PolynomialFeatures(degree=degree),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    )


def evaluate_model(model):
    """Fit the model on the 100 sorted fragments and return its accuracy on them."""
    model.fit(X, y)                     # This fits the model to the data: the computer learns the best boundary.
    predictions = model.predict(X)      # This makes predictions on the data using the fitted model.
    return accuracy_score(y, predictions)


def logistic_regression_degree():
    """A slider for the degree of the polynomial, and the boundary logistic regression learns with it."""
    degree = widgets.IntSlider(value=1, min=1, max=12, description="Degree:", continuous_update=False)
    score = widgets.HTML()
    picture = _picture_widget()

    def update(change=None):
        model = polynomial_logistic_regression(degree.value)
        accuracy = evaluate_model(model)
        fig, ax = plt.subplots(figsize=(7, 5))
        plot_decision_boundary(model, ax)
        plot_dataset(X, y, ax)
        ax.set_title(f"Polynomial degree {degree.value}")
        picture.value = _png(fig)
        score.value = f"<b>Accuracy: {accuracy:.1%}</b>"

    degree.observe(update, names="value")
    update()
    display(widgets.VBox([degree, score, picture]))


MODEL_NAMES = ["Logistic regression", "K-nearest neighbors", "Decision tree", "SVM (RBF)", "Neural network"]


def model_playground(new_data=False):
    """Choose a model and its settings, and see the boundary it learns from the 100 sorted fragments.
    With new_data=True, the other 400 fragments are drawn too, and the model is scored on them as well."""
    choice = widgets.Dropdown(options=MODEL_NAMES, description="Model:")
    slider = dict(continuous_update=False, style={"description_width": "110px"})
    degree = widgets.IntSlider(value=3, min=1, max=12, description="degree", **slider)
    neighbors = widgets.IntSlider(value=5, min=1, max=30, description="neighbors", **slider)
    depth = widgets.IntSlider(value=5, min=1, max=15, description="max depth", **slider)
    svm_c = widgets.FloatLogSlider(value=1, base=10, min=-2, max=3, step=.1, description="C", **slider)
    gamma = widgets.FloatLogSlider(value=.1, base=10, min=-3, max=1, step=.1, description="gamma", **slider)
    layers = widgets.IntSlider(value=2, min=1, max=3, description="hidden layers", **slider)
    neurons = widgets.IntSlider(value=20, min=5, max=60, step=5, description="neurons/layer", **slider)
    settings = {
        "Logistic regression": [degree],
        "K-nearest neighbors": [neighbors],
        "Decision tree": [depth],
        "SVM (RBF)": [svm_c, gamma],
        "Neural network": [layers, neurons],
    }
    controls = [degree, neighbors, depth, svm_c, gamma, layers, neurons]
    score = widgets.HTML()
    picture = _picture_widget()

    def selected_model():
        name = choice.value
        if name == "Logistic regression":
            return polynomial_logistic_regression(degree.value)
        if name == "K-nearest neighbors":
            return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=neighbors.value))
        if name == "Decision tree":
            return DecisionTreeClassifier(max_depth=depth.value, random_state=SEED)
        if name == "SVM (RBF)":
            return make_pipeline(StandardScaler(), SVC(C=svm_c.value, gamma=gamma.value))
        hidden = tuple([neurons.value] * layers.value)
        return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=hidden, max_iter=2000, random_state=SEED))

    def update(change=None):
        for control in controls:
            control.layout.display = "" if control in settings[choice.value] else "none"
        model = selected_model()
        accuracy = evaluate_model(model)
        fig, ax = plt.subplots(figsize=(7, 5))
        plot_decision_boundary(model, ax)
        if new_data:
            plot_dataset(X, y, ax, alpha=0.5, classes=["Original Vase A", "Original Vase B"])
            plot_dataset(X_new, y_new, ax, marker="x", alpha=0.35, classes=["New Vase A", "New Vase B"])
            new_accuracy = accuracy_score(y_new, model.predict(X_new))
            score.value = (f"<b>Accuracy on the original 100 fragments: {accuracy:.1%}</b><br>"
                           f"<b>Accuracy on the new 400 fragments: {new_accuracy:.1%}</b>")
        else:
            plot_dataset(X, y, ax)
            score.value = f"<b>Accuracy: {accuracy:.1%}</b>"
        ax.set_title(choice.value)
        picture.value = _png(fig)

    for control in [choice, *controls]:
        control.observe(update, names="value")
    update()
    display(widgets.VBox([choice, *controls, score, picture]))


# --------------------------------------------------
# Plots drawn in the browser
# --------------------------------------------------
# The two pages below run in the browser, so the plot answers the mouse and the sliders at once, without Python.
# Both share the plotting code in _PLOT_JS. The fragments come in as JSON.

_PLOT_JS = """
function makePlot(root, data) {
  const W = 640, H = 470, M = {l: 48, r: 14, t: 14, b: 42};
  const canvas = root.querySelector("canvas"), ctx = canvas.getContext("2d");
  const dpr = window.devicePixelRatio || 1;
  canvas.width = W * dpr;
  canvas.height = H * dpr;
  ctx.scale(dpr, dpr);
  const [x0, x1] = data.xlim, [y0, y1] = data.ylim;
  const sx = x => M.l + (x - x0) / (x1 - x0) * (W - M.l - M.r);
  const sy = v => H - M.b - (v - y0) / (y1 - y0) * (H - M.t - M.b);

  // A grid of points over the whole plot; each one is coloured by the vase predicted there.
  const NX = 128, NY = 94, grid = [];
  for (let j = 0; j < NY; j++) for (let i = 0; i < NX; i++)
    grid.push([x0 + (i + 0.5) / NX * (x1 - x0), y0 + (j + 0.5) / NY * (y1 - y0)]);

  function ticks(lo, hi) {
    const raw = (hi - lo) / 6, mag = Math.pow(10, Math.floor(Math.log10(raw)));
    const step = [1, 2, 2.5, 5, 10].map(m => m * mag).find(s => s >= raw);
    const out = [];
    for (let v = Math.ceil(lo / step) * step; v <= hi + 1e-9; v += step) out.push(+v.toFixed(6));
    return out;
  }

  // Which side of the boundary a point is on: positive on one side, negative on the other.
  // Both ways of naming the sides are tried, and the one that gets more fragments right is kept.
  function classify(side) {
    const s = data.X.map(side);
    const correct = flip => s.filter((v, k) => (flip ? (v < 0 ? 1 : 0) : (v > 0 ? 1 : 0)) === data.y[k]).length;
    const flip = correct(true) > correct(false);
    const predict = v => flip ? (v < 0 ? 1 : 0) : (v > 0 ? 1 : 0);
    const mistakes = s.map((v, k) => predict(v) !== data.y[k]);
    const wrong = mistakes.filter(Boolean).length;
    return {mistakes, wrong, accuracy: 1 - wrong / data.y.length, regions: grid.map(p => predict(side(p)))};
  }

  function draw(state) {
    ctx.fillStyle = "#fff";
    ctx.fillRect(0, 0, W, H);
    ctx.save();
    ctx.beginPath();
    ctx.rect(M.l, M.t, W - M.l - M.r, H - M.t - M.b);
    ctx.clip();
    if (state.regions) {
      const cw = (W - M.l - M.r) / NX, ch = (H - M.t - M.b) / NY;
      ctx.globalAlpha = 0.18;
      state.regions.forEach((k, n) => {
        ctx.fillStyle = data.colors[k];
        ctx.fillRect(M.l + (n % NX) * cw, H - M.b - (Math.floor(n / NX) + 1) * ch, cw + 0.6, ch + 0.6);
      });
      ctx.globalAlpha = 1;
    }
    ctx.strokeStyle = "#ddd";
    ctx.lineWidth = 1;
    for (const v of ticks(x0, x1)) { ctx.beginPath(); ctx.moveTo(sx(v), M.t); ctx.lineTo(sx(v), H - M.b); ctx.stroke(); }
    for (const v of ticks(y0, y1)) { ctx.beginPath(); ctx.moveTo(M.l, sy(v)); ctx.lineTo(W - M.r, sy(v)); ctx.stroke(); }
    data.X.forEach(([a, b], k) => {
      ctx.globalAlpha = 0.8;
      ctx.fillStyle = data.colors[data.y[k]];
      ctx.beginPath(); ctx.arc(sx(a), sy(b), 4.5, 0, 2 * Math.PI); ctx.fill();
      ctx.globalAlpha = 1;
      if (state.mistakes && state.mistakes[k]) {
        ctx.strokeStyle = "#000"; ctx.lineWidth = 1.5;
        ctx.beginPath(); ctx.arc(sx(a), sy(b), 8, 0, 2 * Math.PI); ctx.stroke();
      }
    });
    if (state.line && state.line.length > 1) {
      ctx.strokeStyle = "#000"; ctx.lineWidth = 2.5; ctx.lineJoin = "round";
      ctx.beginPath();
      state.line.forEach(([a, b], k) => k ? ctx.lineTo(sx(a), sy(b)) : ctx.moveTo(sx(a), sy(b)));
      ctx.stroke();
    }
    ctx.restore();

    // Frame, tick labels and axis names.
    ctx.strokeStyle = "#000"; ctx.lineWidth = 1;
    ctx.strokeRect(M.l, M.t, W - M.l - M.r, H - M.t - M.b);
    ctx.fillStyle = "#000"; ctx.font = "12px sans-serif";
    ctx.textAlign = "center"; ctx.textBaseline = "top";
    for (const v of ticks(x0, x1)) ctx.fillText(v, sx(v), H - M.b + 5);
    ctx.fillText("x", (M.l + W - M.r) / 2, H - 18);
    ctx.textAlign = "right"; ctx.textBaseline = "middle";
    for (const v of ticks(y0, y1)) ctx.fillText(v, M.l - 6, sy(v));
    ctx.fillText("y", 14, (M.t + H - M.b) / 2);

    // Legend.
    const entries = [[0, data.classes[0]], [1, data.classes[1]]];
    if (state.mistakes) entries.push([-1, "Mistakes"]);
    const lx = W - M.r - 100, ly = M.t + 8;
    ctx.globalAlpha = 0.85; ctx.fillStyle = "#fff";
    ctx.fillRect(lx - 8, ly - 4, 102, entries.length * 18 + 6);
    ctx.globalAlpha = 1; ctx.strokeStyle = "#ccc"; ctx.strokeRect(lx - 8, ly - 4, 102, entries.length * 18 + 6);
    ctx.textAlign = "left";
    entries.forEach(([k, text], n) => {
      const cy = ly + 8 + n * 18;
      ctx.beginPath(); ctx.arc(lx + 4, cy, k < 0 ? 6 : 4.5, 0, 2 * Math.PI);
      if (k < 0) { ctx.strokeStyle = "#000"; ctx.lineWidth = 1.5; ctx.stroke(); }
      else { ctx.fillStyle = data.colors[k]; ctx.fill(); }
      ctx.fillStyle = "#000"; ctx.fillText(text, lx + 16, cy);
    });
  }

  // From a mouse or touch event to the place in the dig site under it.
  function toData(e) {
    const box = canvas.getBoundingClientRect();
    const px = (e.clientX - box.left) / box.width * W, py = (e.clientY - box.top) / box.height * H;
    return [x0 + (px - M.l) / (W - M.l - M.r) * (x1 - x0), y0 + (H - M.b - py) / (H - M.t - M.b) * (y1 - y0)];
  }

  return {draw, classify, toData, xlim: data.xlim, ylim: data.ylim, scale: (x1 - x0) / (W - M.l - M.r)};
}
"""

_STYLE = """
  <style>
    #__UID__ { background: #fff; color: #000; padding: 12px; font-family: sans-serif; border-radius: 6px; max-width: 664px; }
    #__UID__ .bar { display: flex; gap: 16px; align-items: center; flex-wrap: wrap; margin-bottom: 8px; font-size: 14px; }
    #__UID__ .bar label { display: flex; gap: 8px; align-items: center; }
    #__UID__ .bar input[type=range] { width: 220px; }
    #__UID__ .bar .value { display: inline-block; min-width: 36px; font-weight: bold; }
    #__UID__ button { font-size: 14px; padding: 4px 14px; border: 1px solid #d08000; background: #f0ad4e; color: #000;
                      border-radius: 4px; cursor: pointer; }
    #__UID__ .score { font-size: 15px; min-height: 2.6em; margin-bottom: 6px; }
    #__UID__ canvas { display: block; width: 640px; max-width: 100%; height: auto; touch-action: none; }
  </style>
"""

# The page draw_your_boundary shows: draw a curve with the mouse, and see how many fragments it puts on the wrong side.
_DRAW_HTML = """
<div id="__UID__">
__STYLE__
  <div class="bar"><button>Reset</button></div>
  <div class="score">Draw a curve and release to evaluate it.</div>
  <canvas style="cursor: crosshair"></canvas>
  <script>
  (() => {
    __PLOT_JS__
    const root = document.getElementById("__UID__");
    const plot = makePlot(root, __DATA__);
    const canvas = root.querySelector("canvas"), score = root.querySelector(".score");
    let line = [], drawing = false;

    // Which side of the drawn curve a point is on: the side of the piece of the curve closest to it.
    function side(p) {
      let best = Infinity, s = 0;
      for (let k = 0; k + 1 < line.length; k++) {
        const [ax, ay] = line[k], dx = line[k + 1][0] - ax, dy = line[k + 1][1] - ay, length = dx * dx + dy * dy;
        if (length < 1e-12) continue;
        const t = Math.max(0, Math.min(1, ((p[0] - ax) * dx + (p[1] - ay) * dy) / length));
        const cx = ax + t * dx, cy = ay + t * dy, d = (p[0] - cx) ** 2 + (p[1] - cy) ** 2;
        if (d < best) { best = d; s = dx * (p[1] - cy) - dy * (p[0] - cx); }
      }
      return s;
    }

    canvas.addEventListener("pointerdown", e => {
      canvas.setPointerCapture(e.pointerId);
      drawing = true;
      line = [plot.toData(e)];
      score.textContent = "Release the mouse to evaluate your boundary.";
      plot.draw({line});
    });
    canvas.addEventListener("pointermove", e => {
      if (!drawing) return;
      const p = plot.toData(e), last = line[line.length - 1];
      if (Math.hypot(p[0] - last[0], p[1] - last[1]) < 3 * plot.scale) return;  // skip moves of under 3 pixels
      line.push(p);
      plot.draw({line});
    });
    const release = () => {
      if (!drawing) return;
      drawing = false;
      if (line.length < 3) {
        score.textContent = "Draw a longer curve.";
        return;
      }
      const result = plot.classify(side);
      plot.draw({line, ...result});
      score.innerHTML = `<b>Accuracy: ${(100 * result.accuracy).toFixed(1)}%</b> · ${result.wrong} mistakes`;
    };
    canvas.addEventListener("pointerup", release);
    canvas.addEventListener("pointercancel", release);
    root.querySelector("button").addEventListener("click", () => {
      line = [];
      score.textContent = "Draw a curve and release to evaluate it.";
      plot.draw({});
    });
    plot.draw({});
  })();
  </script>
</div>
"""

# The page straight_line shows: two sliders set the slope and the intercept of a straight boundary.
_LINE_HTML = """
<div id="__UID__">
__STYLE__
  <div class="bar">
    <label>Slope (m): <input type="range" class="slope" min="-5" max="5" step="0.1" value="0"><span class="value"></span></label>
    <label>Intercept (b): <input type="range" class="intercept" min="__YMIN__" max="__YMAX__" step="0.1" value="0"><span class="value"></span></label>
  </div>
  <div class="score"></div>
  <canvas></canvas>
  <script>
  (() => {
    __PLOT_JS__
    const root = document.getElementById("__UID__");
    const plot = makePlot(root, __DATA__);
    const slope = root.querySelector(".slope"), intercept = root.querySelector(".intercept");
    const score = root.querySelector(".score");

    function update() {
      const m = +slope.value, b = +intercept.value;
      slope.nextElementSibling.textContent = m.toFixed(1);
      intercept.nextElementSibling.textContent = b.toFixed(1);
      const result = plot.classify(p => p[1] - (m * p[0] + b));  // above the line or below it
      const line = plot.xlim.map(x => [x, m * x + b]);
      plot.draw({line, ...result});
      score.innerHTML = `<b>Accuracy: ${(100 * result.accuracy).toFixed(1)}%</b> · ${result.wrong} mistakes` +
                        `<br><small>Decision boundary: y = ${m.toFixed(1)}x ${b < 0 ? "−" : "+"} ${Math.abs(b).toFixed(1)}</small>`;
    }
    slope.addEventListener("input", update);
    intercept.addEventListener("input", update);
    update();
  })();
  </script>
</div>
"""


def _page(template, **values):
    """Fill in a page template and show it."""
    data = {
        "X": np.round(X, 4).tolist(),
        "y": y.tolist(),
        "xlim": [float(v) for v in XLIM],
        "ylim": [float(v) for v in YLIM],
        "colors": CLASS_COLORS,
        "classes": classes,
    }
    page = template.replace("__STYLE__", _STYLE).replace("__PLOT_JS__", _PLOT_JS)
    for key, value in {"__UID__": f"ml-{uuid.uuid4().hex}", "__DATA__": json.dumps(data), **values}.items():
        page = page.replace(key, value)
    display(HTML(page))


def draw_your_boundary():
    """Draw a boundary with the mouse. On release, the accuracy of your boundary is shown and the mistakes are circled."""
    _page(_DRAW_HTML)


def straight_line():
    """A straight boundary, y = m·x + b. Two sliders set m and b."""
    _page(_LINE_HTML, __YMIN__=f"{np.floor(YLIM[0] * 10) / 10:.1f}", __YMAX__=f"{np.ceil(YLIM[1] * 10) / 10:.1f}")

Here are the 100 fragments Marc sorted. Each dot is one fragment at its place in the dig site, coloured by the vase it belongs to.

In [ ]:
show_dataset()

Before using a model, try to make a rule yourself. Run the cell, then click and drag across the plot to draw a boundary with your mouse. The accuracy of your classification using your boundary will be displayed on top of the plot. With **Reset**, you can clear the plot and try again. You can also simply draw a new boundary. 

In [ ]:
draw_your_boundary()

Drawing a boundary is a good way to separate the two classes, but it is manual labor. There is no "machine learning" about it. So instead of drawing a boundary ourself, we would like a computer to learn the best boundary possible. 

First we choose a model and then we instruct the computer to learn the best boundary for that model. Don't worry if you don't know what a model is. For now, you can think of it as a set of rules that the computer can use to separate the two classes. 

The first model we can try is using a straight line to separate the two classes. The two rules that we can control are therefore the slope and the intercept of the line. Run the code below and try to find the best straight line that separates the two classes. You can change the slope and intercept of the line using the sliders.

In [ ]:
straight_line()

Are you satisfied with the results? Hopefully you achieved a good accuracy. The model you have just fitted is an example of **logistic regression**. As you can see, a straight line is not enough to separate the two classes. Luckily, logistic regression can be extended to use a quadratic, qubic, or even higher order polynomial to separate the two classes. Run the code below and change the degree of the polynomial to see how it affects the accuracy.

In [ ]:
logistic_regression_degree()

Congratulations! You have just trained your first machine learning model. By changing the polynomial degree, you changed the complexity of the model. In the background, the computer automatically learned the optimal parameters.

But logistic regression is not the only model out there. In the next section, you can choose between **K-nearest neighbors**, **decision trees**, **support vector machines**, **neural networks**, and of course **logistic regression**. We will learn about all of these models throughout the course, but for now, you can just try them out and see how they perform. Write down the settings that gave you the best accuracy for each model. It will come in handy later on.

Based on your experiments, which model and settings would you recommend to Marc to classify the fragments?

In [ ]:
model_playground()

>For the curious: Decision trees model makes completly vertical or horizontal boundaries. Can you figure out why? 

### After the weekend

It's Monday morning. Based on your recommendations, Marc used the model and settings you suggested to classify the rest of the 400 fragments. Without knowing the true labels of the fragments, he reconstructed the two vases. But when he showed the reconstructed vases to his professor, she was not happy. She said that the vases were not reconstructed correctly and that Marc must have misclassified some of the fragments.

He and his professor then had to manually reclassify all the fragments. Marc later gave you the remaining 400 fragments and their true labels. Play again with the same models and settings you used before and see how well they perform on the new data. Does higher accuracy on the original 100 fragments mean higher accuracy on the new 400 fragments?

In [ ]:
model_playground(new_data=True)